In [1]:
# Setting up RAG

# Set up the OpenAI client:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Load the data and build the search index:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

# Create the assistant:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [2]:
# Testing it

# Let's try a question:
assistant.rag("How do I run Ollama locally?")

# This works fine. The search finds relevant FAQ entries about Ollama, and the LLM gives a good answer.

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system.\n   - macOS: download the `.pkg` and install it\n   - Windows: download the `.msi` and install it\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To check that the local server is running, you can run:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf needed, you can restart the Ollama server with:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [3]:
# Now try something slightly different:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about **Olama** specifically.\n\nIf you meant **running the course locally**, the FAQ says you can do that instead of using Codespaces, as long as you’re comfortable setting up the needed tools such as **Python, `uv`, Jupyter, Docker, and anything else required by the module**. It also says to **document your setup and keep it reproducible**.\n\nIf you meant something else by “Olama,” tell me the exact tool name and I’ll check the FAQ context again.'

In [4]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I still join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Yes—if the course is still open, you can usually still join it.\n\nA couple of things to check:\n- **Enrollment deadline:** Some courses close registration after the first class or after a certain date.\n- **Availability:** There may be a waiting list or limited spots.\n- **Prerequisites/requirements:** Make sure you meet any entry requirements.\n\nIf you want, I can help you draft a quick message to the course organizer like:\n\n> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll at this point? Thank you.\n\nIf you share the course name or platform, I can help you check what to do next.'

In [5]:
# now instead of sending only a question to the lllm well also give him the function he can use

# defining the tool: 

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search('how do i run ollama')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [7]:
# llms are language agnostic. we just making an http call to some endpoint, and we need 
# a language agnostic way of defining a function:

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)


In [9]:
len(response.output)

1

In [10]:
call = response.output[0]

In [11]:
# llm already changed the question (this is not the question in the way we)
call

ResponseFunctionToolCall(arguments='{"query":"Can I still join the course if I just discovered it? late enrollment join after start"}', call_id='call_oHe1y8hxv3Sz8cfXBaBCjJmF', name='search', type='function_call', id='fc_0e58dca57163f80e006a39b0ec568c819e9a658ec7e60b620a', namespace=None, status='completed')

In [12]:
# this was the tool he is using
call.name

'search'

In [13]:
# now we need to parse these arguments

import json

# this is the query he is using 
args = json.loads(call.arguments)

In [14]:
results = search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch

In [15]:
# what happenend?
# history


# 1. make a call to te LLM <-- we pay
# 2. LLM decided to invoke search('params')
# 3. we invoke the search , we have the results 
# 4. send the results back to the llm (another call) <-- we pay again
# 5. llm processes the results 
# 6. llm gives the answer

In [16]:
# -> when we turn rag into agentic rag, we need to do more!

# turn everything into json what well send to the llm

result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

In [17]:
# we received a functionToolCall response, a decision of an llm to use a tool
# basically: he told us to invoke the function, we used the function,  
# there i a related call id (bc we need to know for which call it was needed)

function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [18]:
# llms are stateless! every time they need the entire previous history!

# append decision
messages.append(call)

# append output
messages.append(function_call_output)

print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I still join it?'}, ResponseFunctionToolCall(arguments='{"query":"Can I still join the course if I just discovered it? late enrollment join after start"}', call_id='call_oHe1y8hxv3Sz8cfXBaBCjJmF', name='search', type='function_call', id='fc_0e58dca57163f80e006a39b0ec568c819e9a658ec7e60b620a', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_oHe1y8hxv3Sz8cfXBaBCjJmF', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly wo

In [19]:
# send results back to the llm to process the results 

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [20]:
# what could happen now is: either it actually has an answer for us, or it decides to make another searhch because it doesnt have enough information


print(response.output_text)

Yes — you can still join and start learning anytime.

If you want a certificate, though, you need to submit your project while submissions are still open.


In [21]:
# summary: the path was a bit longer but now the agent is in control and can decide whats happening with the user 
# query and what to do with the too: now the llm is thedriver!

# that is what makes the call to the llm agentic!

# agent has a certain task: (in this case to help the user student)
# agent has memory
# agent has tools (to actually carry out the task)



In [22]:
# how much we payed for this? 

usage = response.usage
print(usage.input_tokens, usage.output_tokens) 

prompt = """ 
i use gpt 5-4 min
(814,35)
this is input output tokens
write a python function to calculate the price and also give the price
"""

819 35


In [23]:
def calculate_gpt54_mini_cost(input_tokens: int, output_tokens: int):
    INPUT_PRICE_PER_MILLION = 0.25   # USD
    OUTPUT_PRICE_PER_MILLION = 2.00  # USD

    input_cost = input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION
    output_cost = output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


cost = calculate_gpt54_mini_cost(814, 35)
print(cost)

{'input_cost': 0.0002035, 'output_cost': 7e-05, 'total_cost': 0.0002735}


In [24]:
# THE AGENTIC LOOP

# What if the LLM decides to make multiple calls?

# what happenend?
# history


# 1. make a call to te LLM <-- we pay
# 2. LLM decided to invoke search('params')
# 3. we invoke the search , we have the results 
# 4. send the results back to the llm (another call) <-- we pay again
# 5. llm processes the results 
# 6. LLM decides to make another tool call
# 7. send one more api request 
# 8. process and gives the answer 
# ...

# for that, we need: 
# 1. developer prompt: the role and behavior
# ...

# -> agent(instructions, memory, tools)

# we exit the loop when there are no more tool calls 


In [25]:
# setup

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)
    # if call.name == "some other fuction"

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]



In [26]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

response.output_text

''

In [27]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join"}', call_id='call_6faFlYHN70eugUJqqusUxMFW', name='search', type='function_call', id='fc_06a4998f300bfca2006a39b0eeb1a481928b008d796b72324b', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"enrollment registration open join course late enrollment"}', call_id='call_JNzVCik6gBCk0qJPyGmj9hUS', name='search', type='function_call', id='fc_06a4998f300bfca2006a39b0eeb1b88192b73df5c3f9c3e24f', namespace=None, status='completed')]

In [28]:
response.output[1]

ResponseFunctionToolCall(arguments='{"query":"enrollment registration open join course late enrollment"}', call_id='call_JNzVCik6gBCk0qJPyGmj9hUS', name='search', type='function_call', id='fc_06a4998f300bfca2006a39b0eeb1b88192b73df5c3f9c3e24f', namespace=None, status='completed')

In [29]:
# add to history (multiple items in the output now)
messages.extend(response.output)

for item in response.output:

    if item.type == "function_call":
        # print("function_call: ", item.name, item.arguments)
        # call_output = make_call(item)
        # messages.append(call_output)

        print("function_call:", item.name, item.arguments)
        # invoke this helper 
        call_output = make_call(item)
        # add to history
        messages.append(call_output)

    elif item.type == "message":
        print("ASSISTANT: ")
        print(item.content[0].text)
        

function_call: search {"query":"join course discovered course can I join"}
function_call: search {"query":"enrollment registration open join course late enrollment"}


In [30]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join"}', call_id='call_6faFlYHN70eugUJqqusUxMFW', name='search', type='function_call', id='fc_06a4998f300bfca2006a39b0eeb1a481928b008d796b72324b', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"enrollment registration open join course late enrollment"}', call_id='call_JNzVCik6gBCk0qJPyGmj9hUS', n

In [31]:
# FOLLOW UP QUESTIONS 

In [32]:
# reset messages 
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

iterations = 1

while True:
    print(f"iteration {iterations}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    # add to history (multiple items in the output now)
    messages.extend(response.output)

    for item in response.output:

        if item.type == "function_call":
            # print("function_call: ", item.name, item.arguments)
            # call_output = make_call(item)
            # messages.append(call_output)

            print("function_call:", item.name, item.arguments)
            # invoke this helper 
            call_output = make_call(item)
            # add to history
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT: ")
            print(item.content[0].text)

    iterations = iterations + 1
    if has_function_calls == False:
        break

        



iteration 1...


function_call: search {"query":"join course late enrollment can I join discovered the course"}
function_call: search {"query":"enroll after course start join course FAQ"}
function_call: search {"query":"register for course after it has started FAQ"}
iteration 2...
ASSISTANT: 
Yes — you can still join. You can start learning and submitting homework while the submission forms are open.

One important caveat: if you want a certificate, you need to submit your project while the course is still accepting submissions. Also, certificates are only available when finishing with a live cohort, not in self-paced mode.

If you’d like, I can also point you to the best place to start in the course materials. Any other areas you want to explore?


In [33]:
# DIFFERENT INSTRUCTIONS 


instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results and then perform more searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

iterations = 1

while True:
    print(f"iteration {iterations}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    # add to history (multiple items in the output now)
    messages.extend(response.output)

    for item in response.output:

        if item.type == "function_call":
            # print("function_call: ", item.name, item.arguments)
            # call_output = make_call(item)
            # messages.append(call_output)

            print("function_call:", item.name, item.arguments)
            # invoke this helper 
            call_output = make_call(item)
            # add to history
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT: ")
            print(item.content[0].text)

    iterations = iterations + 1
    if has_function_calls == False:
        break 


iteration 1...
function_call: search {"query":"join course discovered late enrollment can I join"}
iteration 2...
function_call: search {"query":"certificate project while accepting submissions self paced no certificate live cohort peer-review submissions open"}
iteration 3...
ASSISTANT: 
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still open. Also, certificates are only available for the live cohort, not for self-paced participation.

If you’d like, I can also help you with:
- how to start from scratch,
- whether you can still catch up,
- or what you need for the certificate.

Anything else you want to explore?


In [34]:
# put that into a function 

def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:



    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    iterations = 1

    while True:
        print(f"iteration {iterations}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool],
        )

        # add to history (multiple items in the output now)
        messages.extend(response.output)

        for item in response.output:

            if item.type == "function_call":
                # print("function_call: ", item.name, item.arguments)
                # call_output = make_call(item)
                # messages.append(call_output)

                print("function_call:", item.name, item.arguments)
                # invoke this helper 
                call_output = make_call(item)
                # add to history
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT: ")
                last_answer = item.content[0].text
                print(last_answer)

        iterations = iterations + 1
        if has_function_calls == False:
            break 

    return last_answer

        

In [35]:

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results and then perform more searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

result = agent_loop(
    instructions=instructions,
    question=question
)

iteration 1...
function_call: search {"query":"join course discovered the course can I join enrollment registration access new student"}
iteration 2...
function_call: search {"query":"accepting submissions certificate still join course can I join project submission live cohort self-paced" }
iteration 3...
ASSISTANT: 
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while the course is still accepting submissions. If you’re just learning for now, you can start anytime.

If you want, I can also help with the next steps for getting started.


In [36]:
result

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while the course is still accepting submissions. If you’re just learning for now, you can start anytime.\n\nIf you want, I can also help with the next steps for getting started.'

In [37]:
question = "what's queen gambit?"

result = agent_loop(
    instructions=instructions,
    question=question
)

iteration 1...
function_call: search {"query":"queen gambit chess what is queen's gambit opening"}
iteration 2...
function_call: search {"query":"queen's gambit chess opening definition sacrifice d4 d5 c4"}
iteration 3...
ASSISTANT: 
A **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a temporary sacrifice to try to gain control of the center. The idea is to tempt Black to capture it, then White uses the central space advantage to develop pieces and pressure the center.

There are two main forms:
- **Queen’s Gambit Accepted**: Black takes the pawn on c4.
- **Queen’s Gambit Declined**: Black does not take it and instead supports the center.

If you want, I can also explain **why it’s called a “gambit”** or show a few common variations.


In [38]:
# adjust insstructions again 


instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results and then perform more searches.

The question has to be about thye course or its logistics, offtopic questions shoudnt be answered.
If the serach returns nothing, it's likely an off-topic question. 
If you can't answer the question using FAQ, don't do it yourself. 
Only use the facts from the FAQ database. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "what's queen gambit?"

result = agent_loop(
    instructions=instructions,
    question=question
)



iteration 1...
function_call: search {"query":"queen gambit"}
iteration 2...
function_call: search {"query":"gambit chess opening queen"}
iteration 3...
function_call: search {"query":"queen gambit chess opening FAQ"}
iteration 4...
ASSISTANT: 
I couldn’t find any course FAQ entry about “queen gambit,” so it doesn’t look like a course/logistics question.

If you meant something related to the course, feel free to rephrase it, and I can check again. Are there other areas of the course you want to explore?


In [39]:
# NOW WE WILL USE A FRAMEWORK INSTEAD OF DOING EVERYTHING MANUALLY

In [40]:
!uv add toyaikit

Resolved 127 packages in 37ms
Checked 123 packages in 273ms


In [41]:
# we need imports 

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [42]:
agent_tools = Tools()
# actual tool / description of the tool
agent_tools.add_tool(search, search_tool)

# also annoying to write this description everytime by hand or with chatgpt
# typically we add docstring and typehints instead to the tool function 

In [43]:
# better: 
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [44]:
# now instead of passing the description we do: 

agent_tools = Tools()
agent_tools.add_tool(search)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [45]:
# he mentioned the frameworks briedflyu: 

# openai agents jdk 
# pedantic ai 
# google adk
# langchain? 
# krueger?? 
# i didnt understand everything what he said
# 
#  


In [46]:
# we define the actual agent 

# toyaikit is doing what we did so far, but a little bit more interactive

chat_interface = IPythonChatInterface()  # this displays the output
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [47]:
# we want to do this a little bit more interactive: 

# everytime something happens, for example tool call or a message is display it will invoke the callback to display things.


result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [48]:
result.cost

CostInfo(input_cost=Decimal('0.00270375'), output_cost=Decimal('0.0014715'), total_cost=Decimal('0.00417525'))

In [49]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results and then perform more searches.\n\nThe question has to be about thye course or its logistics, offtopic questions shoudnt be answered.\nIf the serach returns nothing, it's likely an off-topic question. \nIf you can't answer the question using FAQ, don't do it yourself. \nOnly use the facts from the FAQ database. \n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunct

In [50]:
# if we want to continue the conversation?

result2 = runner.loop(
    previous_messages=result.all_messages,
    prompt="How do I run a different model?",
    callback=callback,
)

-> Response received


-> Response received


In [1]:
# Interactive chat
# runner.run();

In [ ]:
# OTHER FRAMEWORKS 

"""
His favorite is PydanticAI. 

he then explained the other ones 

"""


# agent

"""
- has instructions 
- has tools 
- has memory

-> this is how we turn the rigig rag into something more dynamic.

-> we don't always need an agent. maybe sometimes we only need 
smart parsing of a document into another, or we can use only rigid rag

-> we should try first without usin an agent, because 
its takes longer, it needs to explore things, we need frameworks, 
we pay more, and also agents can go wild and do toolcall after toolcall etc 


then we hav to monitor costs and whats happening , so if we dont really need 
this complexity for our project then we try to avoid it

but if we tried all other things and can be confident that we need and 
we can justify the additional complexity and costs
"""